# Единый датасет модулей M1–M5

Все данные подгружаются **напрямую с сайтов** ЦБ, Минфина и ФНС (как в исходных ноутбуках). Локальные файлы и ручной импорт **не нужны**.

**M2** опрашивает страницы репо ЦБ по всем **вторникам** с 2010 года до сегодня; это долго (сотни HTTP-запросов с небольшой паузой). При необходимости уменьшите период в ячейке M2.

Ключ объединения: **`date`**. На днях без аукциона репо признаки M2 остаются NaN.


In [1]:
import re
import time
import xml.etree.ElementTree as ET
from datetime import date, datetime, timedelta
from io import BytesIO

import numpy as np
import pandas as pd
import requests


In [2]:
# --- M1: резервы + RUONIA (ежедневно, m1_fact/m1_need интерполяция на календарь) ---
url_rr = "https://www.cbr.ru/vfs/hd_base/RReserves/required_reserves_table.xlsx"
df_rr = pd.read_excel(url_rr, header=2, skipfooter=8, usecols=[0, 1, 2])
df_rr.columns = ["m1_data", "m1_fact", "m1_need"]
df_rr["m1_data"] = pd.to_datetime(df_rr["m1_data"])

end_m1 = pd.Timestamp.today().normalize()
# ToDate в запросе ЦБ — в формате MM/DD/YYYY (как в исходном M1.ipynb)
url_ruon = (
    "https://www.cbr.ru/Queries/UniDbQuery/DownloadExcel/14315?Posted=True"
    f"&From=11.01.2010&To={end_m1:%d.%m.%Y}"
    "&FromDate=01%2F11%2F2010"
    f"&ToDate={end_m1:%m}%2F{end_m1:%d}%2F{end_m1:%Y}"
    "&backUrl=%2Fhd_base%2Fruonia%2Fdynamics%2F"
)
df2_ruon = pd.read_excel(url_ruon, usecols=[0, 1, 2, 5, 8])
df2_ruon["ruo_data"] = pd.to_datetime(df2_ruon["DT"])

all_dates = pd.Series(
    pd.date_range(
        start=min(df2_ruon["ruo_data"].min(), df_rr["m1_data"].min()),
        end=max(df2_ruon["ruo_data"].max(), df_rr["m1_data"].max()),
        freq="D",
    )
)
df_full = pd.DataFrame({"data": all_dates})
df_full = pd.merge(df_full, df_rr, left_on="data", right_on="m1_data", how="left")
df_full["m1_fact"] = df_full["m1_fact"].interpolate(method="linear")
df_full["m1_need"] = df_full["m1_need"].interpolate(method="linear")
df_full = df_full.drop("m1_data", axis=1)
df_result = pd.merge(df2_ruon, df_full, left_on="ruo_data", right_on="data", how="left")

df_m1 = df_result.copy()
df_m1["m1_shift"] = df_m1["m1_fact"] - df_m1["m1_need"]
df_m1 = df_m1.rename(
    columns={
        "data": "m1_data",
        "ruo": "m1_ruo",
        "vol": "m1_vol",
        "MinRate": "m1_MinRate",
        "MaxRate": "m1_MaxRate",
    }
)
df_m1 = df_m1[
    ["m1_data", "m1_fact", "m1_need", "m1_shift", "m1_ruo", "m1_vol", "m1_MinRate", "m1_MaxRate"]
]

window_days = 3 * 365
epsilon = 1e-9

def get_rolling_mad(series, window):
    return series.rolling(window=window, min_periods=1).apply(
        lambda x: np.median(np.abs(x - np.median(x)))
    )

shift_med = df_m1["m1_shift"].rolling(window=window_days, min_periods=1).median()
shift_mad = get_rolling_mad(df_m1["m1_shift"], window_days)
df_m1["m1_shift_mad"] = (df_m1["m1_shift"] - shift_med) / (shift_mad + epsilon)

ruo_med = df_m1["m1_ruo"].rolling(window=window_days, min_periods=1).median()
ruo_mad = get_rolling_mad(df_m1["m1_ruo"], window_days)
df_m1["m1_ruo_mad"] = (df_m1["m1_ruo"] - ruo_med) / (ruo_mad + epsilon)


def get_days_to_end(dates):
    days = []
    for d in dates:
        if d.day >= 15:
            if d.month == 12:
                end_date = d.replace(year=d.year + 1, month=1, day=14)
            else:
                end_date = d.replace(month=d.month + 1, day=14)
        else:
            end_date = d.replace(month=d.month, day=14)
        days.append((end_date - d).days)
    return days


df_m1["days_to_end"] = get_days_to_end(df_m1["m1_data"])
df_m1["m1_Flag_EndOfPeriod"] = (df_m1["days_to_end"] <= 5).astype(int)
df_m1 = df_m1.drop(columns=["days_to_end"])
df_m1 = df_m1.sort_values("m1_data").reset_index(drop=True)
print("M1:", df_m1.shape, df_m1["m1_data"].min(), "—", df_m1["m1_data"].max())


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


M1: (4018, 11) 2010-01-11 00:00:00 — 2026-05-07 00:00:00


In [3]:
# --- M2: репо 7д с сайта ЦБ (вторники) + ключевая ставка, MAD — как в m2.ipynb ---
# Период сбора (при необходимости сузьте start_date / end_date — меньше запросов к cbr.ru)
start_date_m2 = datetime(2010, 1, 1)
end_date_m2 = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)


def get_auction_data_m2(date_obj):
    """Один день, одна страница hd_base/repo; только 7-дневные аукционы со спросом и размещением."""
    date_str = date_obj.strftime("%d.%m.%Y")
    url = (
        "https://www.cbr.ru/hd_base/repo/?UniDbQuery.Posted=True"
        f"&UniDbQuery.From={date_str}&UniDbQuery.To={date_str}&UniDbQuery.P1=0"
    )
    try:
        tables = pd.read_html(url)
        if not tables or len(tables) == 0:
            return None
        df = tables[0]
        if df.shape[1] < 2:
            return None
        result = {"Дата": date_str}
        for _, row in df.iterrows():
            key = str(row.iloc[0]).strip()
            value = row.iloc[1] if len(row) > 1 else None
            if value in ("—", "-") or pd.isna(value):
                continue
            if "спрос" in key.lower():
                try:
                    val_str = str(value).replace(" ", "").replace(",", ".")
                    result["Спрос"] = float(val_str)
                except Exception:
                    pass
            elif "общий объем заключенных сделок" in key.lower():
                try:
                    val_str = str(value).replace(" ", "").replace(",", ".")
                    result["Размещение"] = float(val_str)
                except Exception:
                    pass
            elif "ставка отсечения" in key.lower():
                try:
                    result["Ставка_отсечения"] = float(value) / 10000
                except Exception:
                    pass
            elif "средневзвешенная ставка" in key.lower() and "лимита" not in key.lower():
                try:
                    result["Средневзвешенная_ставка"] = float(value) / 10000
                except Exception:
                    pass
            elif "срок" in key.lower() and "дни" in key.lower():
                try:
                    result["Срок"] = int(value)
                except Exception:
                    pass
        if result.get("Срок") == 7 and "Спрос" in result and "Размещение" in result:
            return result
        return None
    except Exception:
        return None


tuesdays = []
cur = start_date_m2
while cur <= end_date_m2:
    if cur.weekday() == 1:
        tuesdays.append(cur)
    cur += timedelta(days=1)

auction_data = []
for i, date_obj in enumerate(tuesdays):
    if i > 0 and i % 100 == 0:
        print(f"   M2: проверено {i}/{len(tuesdays)} вторников, найдено аукционов {len(auction_data)}")
    data = get_auction_data_m2(date_obj)
    if data:
        auction_data.append(data)
    time.sleep(0.05)

if not auction_data:
    raise RuntimeError("M2: не удалось собрать ни одного 7-дневного аукциона (проверьте сеть / доступность cbr.ru).")

df_auctions = pd.DataFrame(auction_data)
df_auctions["Дата"] = pd.to_datetime(df_auctions["Дата"], format="%d.%m.%Y")
df_auctions = df_auctions.sort_values("Дата").reset_index(drop=True)
df_auctions["Cover_ratio"] = df_auctions["Спрос"] / df_auctions["Размещение"]
df_auctions_clean = df_auctions[df_auctions["Дата"] >= "2013-01-01"].copy()

url_keyrate = (
    "https://www.cbr.ru/hd_base/keyrate/?UniDbQuery.Posted=True"
    "&UniDbQuery.From=17.09.2013"
    f"&UniDbQuery.To={pd.Timestamp.today():%d.%m.%Y}"
)
tables_kr = pd.read_html(url_keyrate)
df_keyrate = tables_kr[0].copy()
df_keyrate.columns = ["Дата", "Ключевая_ставка"]
df_keyrate["Дата"] = pd.to_datetime(df_keyrate["Дата"], format="%d.%m.%Y")

df_m2 = df_auctions_clean.merge(df_keyrate, on="Дата", how="inner")
df_m2["Rate_spread"] = df_m2["Средневзвешенная_ставка"] - df_m2["Ключевая_ставка"]
df_m2["Flag_Demand"] = (df_m2["Cover_ratio"] > 2.0).astype(int)


def calc_mad_score(df, column, window_days=1095):
    scores = []
    for i in range(len(df)):
        current_date = df["Дата"].iloc[i]
        start_date = current_date - pd.Timedelta(days=window_days)
        mask = (df["Дата"] >= start_date) & (df["Дата"] < current_date)
        window_data = df.loc[mask, column]
        if len(window_data) >= 10:
            median_val = window_data.median()
            mad_val = (window_data - median_val).abs().median()
            score = (df[column].iloc[i] - median_val) / mad_val if mad_val > 0 else 0.0
        else:
            score = 0.0
        scores.append(score)
    return scores


df_m2["MAD_score_cover"] = calc_mad_score(df_m2, "Cover_ratio")
df_m2["MAD_score_rate_spread"] = calc_mad_score(df_m2, "Rate_spread")
df_m2 = df_m2.sort_values("Дата").reset_index(drop=True)
print("M2:", df_m2.shape, df_m2["Дата"].min(), "—", df_m2["Дата"].max(), "| собрано строк репо 7д:", len(df_auctions))


   M2: проверено 100/853 вторников, найдено аукционов 30
   M2: проверено 200/853 вторников, найдено аукционов 51
   M2: проверено 300/853 вторников, найдено аукционов 144
   M2: проверено 400/853 вторников, найдено аукционов 184
   M2: проверено 500/853 вторников, найдено аукционов 184
   M2: проверено 600/853 вторников, найдено аукционов 184
   M2: проверено 700/853 вторников, найдено аукционов 192
   M2: проверено 800/853 вторников, найдено аукционов 195
M2: (198, 12) 2013-09-17 00:00:00 — 2026-05-05 00:00:00 | собрано строк репо 7д: 242


In [4]:
# --- M3: ОФЗ Minfin, дневной ряд (ffill между датами аукционов) ---
headers_m3 = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

urls_by_year = {
    2026: "https://minfin.gov.ru/common/upload/library/2026/05/main/INTERNET_Auction_Results_rus_2026_20260507.xlsx",
    2025: "https://minfin.gov.ru/common/upload/library/2026/01/main/INTERNET_Auction_Results_rus_2025_20251231.xlsx",
    2024: "https://minfin.gov.ru/common/upload/library/2025/01/main/INTERNET_Auction_Results_rus_2024_20241231.xlsx",
    2023: "https://minfin.gov.ru/common/upload/library/2024/01/main/INTERNET_Auction_Results_rus_2023_20231231.xlsx",
    2022: "https://minfin.gov.ru/common/upload/library/2023/01/main/INTERNET_Auction_Results_rus_2022_20221222.xlsx",
    2021: "https://minfin.gov.ru/common/upload/library/2022/02/main/INTERNET_Auction_Results_rus_2021_20211223.xlsx",
    2020: "https://minfin.gov.ru/common/upload/library/2020/12/main/INTERNET_Auction_Results_rus_2020_20201223.xlsx",
    2019: "https://minfin.gov.ru/common/upload/library/2020/01/main/INTERNET_Auction_Results_rus_2019_20191218.xlsx",
    2018: "https://minfin.gov.ru/common/upload/library/2019/01/main/INTERNET_Auction_Results_rus_2018_20181226.xlsx",
    2017: "https://minfin.gov.ru/common/upload/library/2017/12/main/INTERNET_Auction_Results_rus_2017_20171227.xlsx",
    2016: "https://minfin.gov.ru/common/upload/library/2017/01/main/INTERNET_Auction_Results_rus_2016_20161230_1.xlsx",
}

COLS_WITH_FORMAT = [
    "date", "format", "code", "type", "maturity_date", "days_to_maturity",
    "volume_offer", "price_cutoff", "price_avg", "yield_cutoff", "yield_avg",
    "volume_demand", "volume_placement", "volume_revenue", "satisfaction_coef",
]
COLS_NO_FORMAT_14 = [
    "date", "code", "type", "maturity_date", "days_to_maturity",
    "volume_offer", "price_cutoff", "price_avg", "yield_cutoff", "yield_avg",
    "volume_demand", "volume_placement", "volume_revenue", "satisfaction_coef",
]
COLS_NO_FORMAT_13 = COLS_NO_FORMAT_14[:13]


def _second_col_is_format(df):
    if df.shape[1] < 2:
        return False
    s = df.iloc[:, 1].dropna().astype(str).str.strip()
    if len(s) == 0:
        return False
    return bool((s.isin(["Аукцион", "ДРПА"])).mean() >= 0.25)


def normalize_minfin_ofz_dataframe(df, year):
    n = df.shape[1]
    has_fmt = _second_col_is_format(df)
    if has_fmt and n == 15:
        df = df.copy()
        df.columns = COLS_WITH_FORMAT
        return df
    if not has_fmt and n == 14:
        df = df.copy()
        df.columns = COLS_NO_FORMAT_14
        df.insert(1, "format", "Аукцион")
        return df
    if not has_fmt and n == 15:
        df = df.iloc[:, :13].copy()
        df.columns = COLS_NO_FORMAT_13
        df.insert(1, "format", "Аукцион")
        df["satisfaction_coef"] = np.nan
        return df
    raise ValueError(f"Год {year}: неожиданная таблица (ncols={n}, format_col={has_fmt})")


def load_ofz_year(year, url):
    r = requests.get(url, headers=headers_m3, timeout=60)
    r.raise_for_status()
    df_year = pd.read_excel(BytesIO(r.content), skiprows=8)
    return normalize_minfin_ofz_dataframe(df_year, year)


all_dfs = [load_ofz_year(y, u) for y, u in sorted(urls_by_year.items(), reverse=True)]
df_all = pd.concat(all_dfs, ignore_index=True)

df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
df_all["maturity_date"] = pd.to_datetime(df_all["maturity_date"], errors="coerce")
for col in [
    "days_to_maturity", "volume_offer", "volume_demand", "volume_placement",
    "price_cutoff", "price_avg", "yield_cutoff", "yield_avg",
    "volume_revenue", "satisfaction_coef",
]:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

m = df_all["satisfaction_coef"].isna() & (df_all["volume_demand"] > 0)
df_all.loc[m, "satisfaction_coef"] = (
    df_all.loc[m, "volume_placement"] / df_all.loc[m, "volume_demand"]
)

df_all = df_all[df_all["format"] == "Аукцион"].copy()
df_all = df_all.sort_values("date").reset_index(drop=True)
for c, newc in [
    ("volume_offer", "volume_offer_bln"),
    ("volume_demand", "volume_demand_bln"),
    ("volume_placement", "volume_placement_bln"),
]:
    df_all[newc] = df_all[c] / 1000.0

df_all_clean = df_all.copy()
df_all_clean = df_all_clean[(df_all_clean["volume_offer"] >= 0) & (df_all_clean["volume_demand"] >= 0)]
df_all_clean = df_all_clean[df_all_clean["volume_offer"] > 0]
df_all_clean["cover_ratio_temp"] = df_all_clean["volume_demand"] / df_all_clean["volume_offer"]
df_all_clean = df_all_clean[df_all_clean["cover_ratio_temp"] <= 100]
df_all_clean = df_all_clean.drop(columns=["cover_ratio_temp"])

df_all = df_all_clean.copy()
df_all["cover_ratio"] = np.where(
    df_all["volume_offer"] > 0,
    df_all["volume_demand"] / df_all["volume_offer"],
    np.nan,
)
df_all["flag_nedospros"] = (df_all["cover_ratio"] < 1.2).astype(int)
df_all["flag_perespros"] = (df_all["cover_ratio"] > 2.0).astype(int)


def yield_spread_vs_nearby_curve(df):
    out = pd.Series(np.nan, index=df.index)

    def _interp(d, y, i, n):
        eps = 1e-9
        if i == 0:
            den = d[2] - d[1]
            return y[1] + (y[2] - y[1]) * (d[0] - d[1]) / (den if abs(den) > eps else eps)
        if i == n - 1:
            den = d[n - 2] - d[n - 3]
            return y[n - 3] + (y[n - 2] - y[n - 3]) * (d[n - 1] - d[n - 3]) / (den if abs(den) > eps else eps)
        den = d[i + 1] - d[i - 1]
        return y[i - 1] + (y[i + 1] - y[i - 1]) * (d[i] - d[i - 1]) / (den if abs(den) > eps else eps)

    def block_spread(g):
        g = g.sort_values("days_to_maturity")
        d = g["days_to_maturity"].to_numpy(dtype=float)
        y = g["yield_avg"].to_numpy(dtype=float)
        n = len(g)
        sp = np.zeros(n)
        if n == 1:
            sp[:] = 0.0
        elif n == 2:
            sp[0] = y[0] - y[1]
            sp[1] = y[1] - y[0]
        else:
            for i in range(n):
                yhat = _interp(d, y, i, n)
                sp[i] = y[i] - yhat
        return pd.Series(sp, index=g.index)

    for _, g in df.groupby("date"):
        out.loc[g.index] = block_spread(g)
    return out


df_all["yield_spread"] = yield_spread_vs_nearby_curve(df_all)
df_all["flag_failed"] = ((df_all["volume_placement"] == 0) | (df_all["volume_placement"].isna())).astype(int)
df_all = df_all.sort_values("date").reset_index(drop=True)

rows = []
for dt, g in df_all.groupby("date"):
    vo = g["volume_offer"].sum()
    vd = g["volume_demand"].sum()
    vp = g["volume_placement"].sum()
    w = g["volume_placement"].fillna(0).clip(lower=0)
    ybar = float(np.average(g["yield_avg"], weights=w)) if w.sum() > 0 else float(g["yield_avg"].mean())
    ys = float(g["yield_spread"].mean())
    cr = vd / vo if vo > 0 else np.nan
    rows.append({
        "date": dt,
        "volume_offer": vo,
        "volume_demand": vd,
        "volume_placement": vp,
        "yield_avg": ybar,
        "yield_spread": ys,
        "cover_ratio": cr,
        "flag_nedospros": int((g["cover_ratio"] < 1.2).any()),
        "flag_perespros": int((g["cover_ratio"] > 2.0).any()),
        "flag_failed": int(g["flag_failed"].max()),
    })

df_grouped = pd.DataFrame(rows)
flag_cols = ["flag_nedospros", "flag_perespros", "flag_failed"]
df_daily = df_grouped.set_index("date").resample("D").ffill()
for flag in flag_cols:
    df_daily[flag] = df_daily[flag].fillna(0).astype(int)
df_daily = df_daily.reset_index()


def mad_normalize_series(series, window_days=756):
    normalized = series.copy()
    for i in range(len(series)):
        start = max(0, i - window_days)
        window_data = series.iloc[start:i].dropna()
        if len(window_data) < 30:
            normalized.iloc[i] = 0.0
            continue
        median = window_data.median()
        mad = (window_data - median).abs().median()
        if mad == 0:
            normalized.iloc[i] = 0.0
        else:
            normalized.iloc[i] = (series.iloc[i] - median) / mad
    return normalized


df_daily_full = df_daily.copy()
df_daily_recent = df_daily[df_daily["date"] >= "2021-01-01"].copy()
cover_ratio_clipped = df_daily_recent["cover_ratio"].clip(upper=10)
df_daily_recent = df_daily_recent.copy()
df_daily_recent["mad_score_cover"] = mad_normalize_series(cover_ratio_clipped)
df_daily_recent["mad_score_yield"] = mad_normalize_series(df_daily_recent["yield_avg"])
df_daily_recent["mad_score_yield_spread"] = mad_normalize_series(df_daily_recent["yield_spread"])

df_daily = df_daily_full.copy()
df_daily.loc[df_daily["date"] >= "2021-01-01", "mad_score_cover"] = df_daily_recent["mad_score_cover"].values
df_daily.loc[df_daily["date"] >= "2021-01-01", "mad_score_yield"] = df_daily_recent["mad_score_yield"].values
df_daily.loc[df_daily["date"] >= "2021-01-01", "mad_score_yield_spread"] = df_daily_recent[
    "mad_score_yield_spread"
].values
df_daily["mad_score_cover"] = df_daily["mad_score_cover"].fillna(0)
df_daily["mad_score_yield"] = df_daily["mad_score_yield"].fillna(0)
df_daily["mad_score_yield_spread"] = df_daily["mad_score_yield_spread"].fillna(0)

df_m3 = df_daily[[
    "date",
    "volume_offer",
    "volume_demand",
    "volume_placement",
    "cover_ratio",
    "yield_avg",
    "yield_spread",
    "flag_nedospros",
    "flag_perespros",
    "mad_score_cover",
    "mad_score_yield",
    "mad_score_yield_spread",
]].copy()
df_m3 = df_m3.sort_values("date").reset_index(drop=True)
print("M3:", df_m3.shape, df_m3["date"].min(), "—", df_m3["date"].max())


M3: (3767, 12) 2016-01-13 00:00:00 — 2026-05-06 00:00:00


In [5]:
# --- M4: календарь ФНС (фиксированный XML), dummy по важным налогам ---
DATA_URL_M4 = "https://data.nalog.ru/opendata/7707329152-kalendar/data-20241231-structure-20140228.xml"

r_cal = requests.get(DATA_URL_M4, timeout=60)
r_cal.raise_for_status()
root_m4 = ET.fromstring(r_cal.content, parser=ET.XMLParser(encoding="windows-1251"))

_STRONG_RE = re.compile(r"<strong>(.*?)</strong>", re.DOTALL | re.IGNORECASE)


def extract_strong_headers(html_fragment: str) -> list[str]:
    out = []
    for m in _STRONG_RE.finditer(html_fragment or ""):
        inner = re.sub(r"<[^>]+>", " ", m.group(1))
        inner = re.sub(r"\s+", " ", inner).strip().rstrip(":").strip()
        if inner:
            out.append(inner)
    return out


MONTH_NUM = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}


def calendar_to_records(root: ET.Element) -> list[dict]:
    rows = []
    for year_el in root.findall("year"):
        y = int(year_el.get("index", "0"))
        for month_el in year_el.findall("month"):
            mkey = (month_el.get("name") or "").lower()
            m = MONTH_NUM.get(mkey)
            if m is None:
                continue
            for day_el in month_el.findall("day"):
                d = int(day_el.get("num", "0"))
                try:
                    dt = date(y, m, d)
                except ValueError:
                    continue
                raw_html = day_el.text or ""
                strongs = extract_strong_headers(raw_html)
                rows.append({
                    "date": dt,
                    "year": y,
                    "month": m,
                    "day": d,
                    "day_type": day_el.get("type"),
                    "n_events_html": len(strongs),
                    "raw_headers": strongs,
                })
    return rows


IMPORTANT_TAX_RULES: list[tuple[str, re.Pattern]] = [
    ("НДФЛ", re.compile(r"Налог на доходы физических лиц")),
    ("НДС", re.compile(r"^НДС\b")),
    ("Страховые взносы", re.compile(r"Страховые взносы на обязательное социальное")),
    ("Страхование от НС и ПЗ", re.compile(r"Страхование от несчастных случаев на производстве")),
    ("Налог на прибыль", re.compile(r"Налог на прибыль организаций")),
    ("НДПИ", re.compile(r"Налог на добычу полезных ископаемых")),
    ("Акцизы", re.compile(r"^Акцизы\b")),
    ("ЕНП", re.compile(r"Единый налоговый платеж")),
    ("Косвенные налоги", re.compile(r"^Косвенные налоги\b")),
    ("Земельный налог", re.compile(r"^Земельный налог\b")),
    ("Налог на имущество организаций", re.compile(r"Налог на имущество организаций")),
    ("Налог на имущество физлиц", re.compile(r"Налог на имущество физических лиц")),
    ("Транспортный налог", re.compile(r"^Транспортный налог\b")),
    ("УСН", re.compile(r"Упрощенная система налогообложения")),
    ("НПД", re.compile(r"Налог на профессиональный доход")),
    ("ЕСХН", re.compile(r"Единый сельскохозяйственный налог")),
    ("Туристический налог", re.compile(r"Туристический налог")),
    ("Торговый сбор", re.compile(r"Торговый сбор")),
    ("Водный налог", re.compile(r"^Водный налог\b")),
    ("Плата за НВОС", re.compile(r"Плата за негативное воздействие на окружающую среду")),
    ("Экологический сбор", re.compile(r"Экологический сбор")),
]


def headers_to_canonical(headers: list[str]) -> list[str]:
    found: set[str] = set()
    order: list[str] = []
    for h in headers:
        for canon, pat in IMPORTANT_TAX_RULES:
            if pat.search(h):
                if canon not in found:
                    found.add(canon)
                    order.append(canon)
                break
    return order


rec_m4 = calendar_to_records(root_m4)
df_days_m4 = pd.DataFrame(rec_m4)
df_days_m4["important_taxes"] = df_days_m4["raw_headers"].apply(headers_to_canonical)
df_days_m4["n_important"] = df_days_m4["important_taxes"].str.len()

exploded_m4 = df_days_m4.explode("important_taxes").dropna(subset=["important_taxes"])
exploded_m4 = exploded_m4.rename(columns={"important_taxes": "tax"})
tax_dummies = pd.crosstab(exploded_m4["date"], exploded_m4["tax"]).astype(np.int8).reindex(
    df_days_m4["date"], fill_value=0
)
feature_m4 = df_days_m4[["date", "year", "month", "day", "day_type", "n_events_html", "n_important"]].set_index(
    "date"
).join(tax_dummies)
feature_m4 = feature_m4.reset_index()
feature_m4["date"] = pd.to_datetime(feature_m4["date"])
# категориальный day_type -> число
feature_m4["m4_day_type_code"] = feature_m4["day_type"].astype("category").cat.codes
feature_m4 = feature_m4.drop(columns=["day_type"])
df_m4 = feature_m4.rename(columns=lambda c: f"m4_{c}" if (c != "date" and not str(c).startswith("m4_")) else c)
print("M4:", df_m4.shape, df_m4["date"].min(), "—", df_m4["date"].max())


M4: (365, 28) 2025-01-01 00:00:00 — 2025-12-31 00:00:00


In [6]:
# --- M5: bliquidity ЦБ (ежедневно) ---
import io as _io


def make_session() -> requests.Session:
    s = requests.Session()
    s.trust_env = False
    s.headers.update({"User-Agent": "Mozilla/5.0"})
    return s


def flatten_columns(columns: pd.Index) -> list[str]:
    out = []
    for col in columns:
        if isinstance(col, tuple):
            parts = []
            for p in col:
                p = str(p).strip()
                if p and not p.startswith("Unnamed") and p not in parts:
                    parts.append(p)
            out.append(" | ".join(parts))
        else:
            out.append(str(col).strip())
    return out


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )


def mad_score_m5(series: pd.Series, window: int = 756) -> pd.Series:
    min_p = max(30, window // 10)
    med = series.rolling(window=window, min_periods=min_p).median()
    mad = (series - med).abs().rolling(window=window, min_periods=min_p).median()
    return (series - med) / (1.4826 * mad.replace(0, np.nan))


def parse_m5_data(from_date: str = "2014-01-01") -> pd.DataFrame:
    base_url = "https://www.cbr.ru/hd_base/bliquidity/"
    from_dt = pd.Timestamp(from_date)
    to_dt = pd.Timestamp(datetime.now().date())
    url_m5 = (
        f"{base_url}?UniDbQuery.Posted=True"
        f"&UniDbQuery.From={from_dt:%d.%m.%Y}"
        f"&UniDbQuery.To={to_dt:%d.%m.%Y}"
    )
    s = make_session()
    html = s.get(url_m5, timeout=60).text
    dff = pd.read_html(_io.StringIO(html), header=[0, 1, 2, 3, 4])[0]
    dff.columns = flatten_columns(dff.columns)
    if dff.shape[1] < 15:
        raise RuntimeError(f"Unexpected CBR table shape: {dff.shape}")
    date_col = dff.columns[0]
    col_def = dff.columns[1]
    col_def_wo = dff.columns[2]
    col_corr = dff.columns[13]
    col_res = dff.columns[14]
    dff = dff[dff[date_col].astype(str).str.contains(r"\d{2}\.\d{2}\.\d{4}", regex=True)].copy()
    dff[date_col] = pd.to_datetime(dff[date_col], dayfirst=True, errors="coerce")
    dff = dff.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    for c in [col_def, col_def_wo, col_corr, col_res]:
        dff[c] = to_num(dff[c])
    m5 = pd.DataFrame({
        "date": dff[date_col],
        "liquidity_deficit": dff[col_def],
        "liquidity_deficit_ex_corr": dff[col_def_wo],
        "corr_accounts": dff[col_corr],
        "required_reserves": dff[col_res],
    })
    m5 = m5[m5["date"] >= from_dt].copy()
    m5["reserve_buffer"] = m5["corr_accounts"] - m5["required_reserves"]
    m5["treasury_pressure"] = m5["liquidity_deficit_ex_corr"].diff()
    m5["treasury_pressure_ma7"] = m5["treasury_pressure"].rolling(7, min_periods=3).mean()
    m5["MAD_score_treasury_pressure"] = mad_score_m5(m5["treasury_pressure"], window=756)
    m5["MAD_score_liquidity_deficit"] = mad_score_m5(m5["liquidity_deficit"], window=756)
    m5["is_month_end"] = m5["date"].dt.is_month_end.astype(int)
    m5["flag_stress"] = (
        (m5["MAD_score_treasury_pressure"] > 2.5)
        | ((m5["MAD_score_treasury_pressure"] > 1.5) & (m5["is_month_end"] == 1))
    ).astype(int)
    return m5.reset_index(drop=True)


df_m5 = parse_m5_data("2014-01-01")
df_m5 = df_m5.rename(columns={c: f"m5_{c}" for c in df_m5.columns if c != "date"})
print("M5:", df_m5.shape, df_m5["date"].min(), "—", df_m5["date"].max())


M5: (3095, 12) 2014-01-01 00:00:00 — 2026-05-08 00:00:00


In [7]:
# --- Объединение: единый календарь, left join модулей ---
def norm_date(s: pd.Series) -> pd.Series:
    t = pd.to_datetime(s, errors="coerce")
    if pd.api.types.is_datetime64tz_dtype(t):
        t = t.dt.tz_convert("UTC").dt.tz_localize(None)
    return t.dt.normalize()


def dedupe_by_date(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if "date" not in df.columns:
        return df
    dup = df["date"].duplicated(keep=False)
    if dup.any():
        print(f"  dedupe {name}: {dup.sum()} строк с повтором date → keep='last'")
    return df.drop_duplicates(subset=["date"], keep="last").sort_values("date").reset_index(drop=True)


df_m1["date"] = norm_date(df_m1["m1_data"])
df_m1 = df_m1.drop(columns=["m1_data"])
df_m1 = dedupe_by_date(df_m1, "M1")

df_m2["date"] = norm_date(df_m2["Дата"])
df_m2 = df_m2.drop(columns=["Дата"])
df_m2.columns = ["m2_" + c if c != "date" else c for c in df_m2.columns]
df_m2 = dedupe_by_date(df_m2, "M2")

df_m3["date"] = norm_date(df_m3["date"])
df_m3.columns = ["m3_" + c if c != "date" else c for c in df_m3.columns]
df_m3 = dedupe_by_date(df_m3, "M3")

df_m4["date"] = norm_date(df_m4["date"])
df_m4 = dedupe_by_date(df_m4, "M4")

df_m5["date"] = norm_date(df_m5["date"])
df_m5 = dedupe_by_date(df_m5, "M5")

dmin = min(
    df_m1["date"].min(),
    df_m2["date"].min(),
    df_m3["date"].min(),
    df_m4["date"].min(),
    df_m5["date"].min(),
)
dmax = max(
    df_m1["date"].max(),
    df_m2["date"].max(),
    df_m3["date"].max(),
    df_m4["date"].max(),
    df_m5["date"].max(),
)
base = pd.DataFrame({"date": pd.date_range(dmin, dmax, freq="D")})
assert base["date"].is_unique

wide = base.merge(df_m1, on="date", how="left", validate="one_to_one")
wide = wide.merge(df_m2, on="date", how="left", validate="one_to_one")
wide = wide.merge(df_m3, on="date", how="left", validate="one_to_one")
wide = wide.merge(df_m4, on="date", how="left", validate="one_to_one")
wide = wide.merge(df_m5, on="date", how="left", validate="one_to_one")

assert wide["date"].is_unique, "После merge даты должны быть уникальны."
wide = wide.sort_values("date").reset_index(drop=True)

if "m4_m4_day_type_code" in wide.columns:
    wide = wide.rename(columns={"m4_m4_day_type_code": "m4_day_type_code"})

# Календарь Y/M/D всегда из date — без пропусков и рассинхрона с merge
wide["m4_year"] = wide["date"].dt.year
wide["m4_month"] = wide["date"].dt.month
wide["m4_day"] = wide["date"].dt.day

for prefix in ("m1_", "m2_", "m3_", "m5_"):
    cols = [c for c in wide.columns if c.startswith(prefix)]
    if cols:
        wide[cols] = wide[cols].ffill()

m4rest = [
    c for c in wide.columns
    if c.startswith("m4_") and c not in ("m4_year", "m4_month", "m4_day")
]
if m4rest:
    wide[m4rest] = wide[m4rest].ffill()
    for c in m4rest:
        wide[c] = pd.to_numeric(wide[c], errors="coerce").fillna(0)

print("Wide table:", wide.shape)
print("Период:", wide["date"].min().date(), "—", wide["date"].max().date())
mcols = [c for c in wide.columns if c.startswith(("m1_", "m2_", "m3_", "m4_", "m5_"))]
if mcols:
    mx = wide[mcols].isna().mean().max()
    print(f"Макс. доля NaN по m*-столбцам после выравнивания: {mx:.6f}")
    na_cols = wide[mcols].isna().mean().sort_values(ascending=False)
    tail = na_cols[na_cols > 0]
    if len(tail):
        print("Столбцы с оставшимися NaN (обычно «хвост» до первой даты модуля):")
        print(tail.head(15).to_string())
    else:
        print("Все m*-столбцы без NaN.")



Wide table: (5962, 71)
Период: 2010-01-11 — 2026-05-08


In [8]:
# --- Анализ пропусков (по столбцам) ---
na_share = wide.isna().mean().sort_values(ascending=False)
na_cnt = wide.isna().sum()
summary = pd.DataFrame({"na_count": na_cnt, "na_share": na_share})
print("Топ-30 столбцов по доле NaN:")
print(summary.head(30).to_string())

print("Строки без единого признака модулей (все m*-столбцы NaN) — оценка:")
mcols = [c for c in wide.columns if c.startswith(("m1_", "m2_", "m3_", "m4_", "m5_"))]
if mcols:
    row_any = wide[mcols].notna().any(axis=1)
    print(f"Строк с хотя бы одним признаком: {row_any.sum()} / {len(wide)}")


Топ-30 столбцов по доле NaN:
                            na_count  na_share
date                               0  0.000000
m1_Flag_EndOfPeriod             1944  0.326065
m1_MaxRate                      4660  0.781617
m1_MinRate                      4660  0.781617
m1_fact                         1944  0.326065
m1_need                         1944  0.326065
m1_ruo                          1944  0.326065
m1_ruo_mad                      1944  0.326065
m1_shift                        1944  0.326065
m1_shift_mad                    1944  0.326065
m1_vol                          1961  0.328916
m2_Cover_ratio                  5764  0.966790
m2_Flag_Demand                  5764  0.966790
m2_MAD_score_cover              5764  0.966790
m2_MAD_score_rate_spread        5764  0.966790
m2_Rate_spread                  5764  0.966790
m2_Ключевая_ставка              5764  0.966790
m2_Размещение                   5764  0.966790
m2_Спрос                        5764  0.966790
m2_Средневзвешенная_ставка     

In [9]:
# --- Первые строки широкой таблицы ---
pd.set_option("display.max_columns", 40)
print(wide.head().to_string())


        date     m1_fact     m1_need    m1_shift  m1_ruo  m1_vol  m1_MinRate  m1_MaxRate  m1_shift_mad  m1_ruo_mad  m1_Flag_EndOfPeriod  m2_Спрос  m2_Размещение  m2_Средневзвешенная_ставка  m2_Срок  m2_Ставка_отсечения  m2_Cover_ratio  m2_Ключевая_ставка  m2_Rate_spread  m2_Flag_Demand  m2_MAD_score_cover  m2_MAD_score_rate_spread  m3_volume_offer  m3_volume_demand  m3_volume_placement  m3_cover_ratio  m3_yield_avg  m3_yield_spread  m3_flag_nedospros  m3_flag_perespros  m3_mad_score_cover  m3_mad_score_yield  m3_mad_score_yield_spread  m4_year  m4_month  m4_day  m4_n_events_html  m4_n_important  m4_Акцизы  m4_Водный налог  m4_ЕНП  m4_ЕСХН  m4_Земельный налог  m4_Косвенные налоги  m4_НДПИ  m4_НДС  m4_НДФЛ  m4_НПД  m4_Налог на имущество организаций  m4_Налог на имущество физлиц  m4_Налог на прибыль  m4_Плата за НВОС  m4_Страхование от НС и ПЗ  m4_Страховые взносы  m4_Торговый сбор  m4_Транспортный налог  m4_Туристический налог  m4_УСН  m4_Экологический сбор  m4_m4_day_type_code  m5_liqui

## Агрегационный слой: LSI (Liquidity Stress Index, 0–100)

**Идея:** отдельной размеченной цели «истинного LSI» нет, поэтому для обучения бустингов строится **интерпретируемая опорная метка (teacher)** — выпуклая комбинация нормализованных MAD-сигналов и флагов M1–M5. Ансамбль бустингов учится **нелинейной агрегации** всех признаков широкой таблицы; выход усредняется и ограничивается **[0, 100]** (`LSI_ensemble`).

**Интерпретируемость:** усреднённые `feature_importances_` по моделям + явный список колонок в `X`.

M4: добавляются **`Tax_Week_Flag`** (окно 7 дней по `m4_n_important`) и **`Seasonal_Factor_sin/cos`** (день года). Флаг **Budget drain** в терминах ТЗ соответствует **`m5_flag_stress`**.


In [10]:
# --- Подготовка признаков и teacher-цели LSI [0, 100] ---
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

wide_lsi = wide.sort_values("date").reset_index(drop=True).copy()

# M4: Tax_Week_Flag, сезонность
if "m4_n_important" in wide_lsi.columns:
    tw = wide_lsi["m4_n_important"].fillna(0).astype(float)
    wide_lsi["Tax_Week_Flag"] = (tw.rolling(7, min_periods=1).max() > 0).astype(np.int8)
else:
    wide_lsi["Tax_Week_Flag"] = np.int8(0)

doy = wide_lsi["date"].dt.dayofyear.astype(float)
wide_lsi["Seasonal_Factor_sin"] = np.sin(2 * np.pi * doy / 365.25)
wide_lsi["Seasonal_Factor_cos"] = np.cos(2 * np.pi * doy / 365.25)

CORE = [
    "m1_shift_mad", "m1_ruo_mad", "m1_Flag_EndOfPeriod",
    "m2_MAD_score_cover", "m2_MAD_score_rate_spread", "m2_Flag_Demand",
    "m3_mad_score_cover", "m3_mad_score_yield_spread",
    "m3_flag_nedospros", "m3_flag_perespros",
    "Tax_Week_Flag", "Seasonal_Factor_sin", "Seasonal_Factor_cos",
    "m5_MAD_score_treasury_pressure", "m5_MAD_score_liquidity_deficit", "m5_flag_stress",
]

num_all = wide_lsi.select_dtypes(include=[np.number]).columns.tolist()
extra = [c for c in num_all if c not in CORE and "date" not in str(c).lower()]
extra = [c for c in extra if not str(c).startswith("LSI")]

feature_cols = []
seen = set()
for c in CORE + sorted(extra):
    if c in wide_lsi.columns and c not in seen:
        feature_cols.append(c)
        seen.add(c)

X = wide_lsi[feature_cols].copy()


def _relu_mad(s: pd.Series, cap: float = 4.0) -> np.ndarray:
    v = np.asarray(s.fillna(0.0), dtype=float)
    v = np.clip(v, 0.0, cap) / cap
    return v


t_parts = []
w = []


def add_term(col, weight=1.0, cap=4.0):
    if col in wide_lsi.columns:
        t_parts.append(_relu_mad(wide_lsi[col], cap) * weight)
        w.append(weight)


add_term("m1_shift_mad", 1.0)
add_term("m1_ruo_mad", 1.0)
add_term("m2_MAD_score_cover", 1.0, cap=8.0)
add_term("m2_MAD_score_rate_spread", 1.0, cap=8.0)
add_term("m3_mad_score_cover", 1.0)
add_term("m3_mad_score_yield_spread", 1.0)
add_term("m5_MAD_score_treasury_pressure", 1.2, cap=6.0)
add_term("m5_MAD_score_liquidity_deficit", 1.0, cap=6.0)

flags = []
if "m1_Flag_EndOfPeriod" in wide_lsi.columns:
    flags.append(wide_lsi["m1_Flag_EndOfPeriod"].fillna(0).astype(float) * 0.08)
if "m2_Flag_Demand" in wide_lsi.columns:
    flags.append(wide_lsi["m2_Flag_Demand"].fillna(0).astype(float) * 0.10)
if "m3_flag_nedospros" in wide_lsi.columns:
    flags.append(wide_lsi["m3_flag_nedospros"].fillna(0).astype(float) * 0.10)
if "m5_flag_stress" in wide_lsi.columns:
    flags.append(wide_lsi["m5_flag_stress"].fillna(0).astype(float) * 0.12)
if "Tax_Week_Flag" in wide_lsi.columns:
    flags.append(wide_lsi["Tax_Week_Flag"].fillna(0).astype(float) * 0.05)

base = np.sum(t_parts, axis=0) / max(sum(w), 1e-9)
flag_bonus = np.sum(flags, axis=0) if flags else 0.0
raw = np.clip(base + flag_bonus, 0.0, None)
wide_lsi["_raw_stress_teacher"] = raw

print("Признаков в X:", len(feature_cols))
print("Первые 30:", feature_cols[:30])


Признаков в X: 73
Первые 30: ['m1_shift_mad', 'm1_ruo_mad', 'm1_Flag_EndOfPeriod', 'm2_MAD_score_cover', 'm2_MAD_score_rate_spread', 'm2_Flag_Demand', 'm3_mad_score_cover', 'm3_mad_score_yield_spread', 'm3_flag_nedospros', 'm3_flag_perespros', 'Tax_Week_Flag', 'Seasonal_Factor_sin', 'Seasonal_Factor_cos', 'm5_MAD_score_treasury_pressure', 'm5_MAD_score_liquidity_deficit', 'm5_flag_stress', 'm1_MaxRate', 'm1_MinRate', 'm1_fact', 'm1_need', 'm1_ruo', 'm1_shift', 'm1_vol', 'm2_Cover_ratio', 'm2_Rate_spread', 'm2_Ключевая_ставка', 'm2_Размещение', 'm2_Спрос', 'm2_Средневзвешенная_ставка', 'm2_Срок']


In [11]:
# --- Ансамбль бустингов + обучение (временной сплит 80/20) ---
y_raw = wide_lsi["_raw_stress_teacher"].values
key_sig = [
    "m1_shift_mad", "m2_MAD_score_cover", "m3_mad_score_cover", "m5_MAD_score_treasury_pressure",
]
if all(c in wide_lsi.columns for c in key_sig):
    present = wide_lsi[key_sig].notna().any(axis=1)
else:
    present = pd.Series(True, index=wide_lsi.index)

X_fit = X.loc[present].copy()
y_raw_fit = y_raw[present.values]
dates_fit = wide_lsi.loc[present, "date"].reset_index(drop=True)

order = np.argsort(dates_fit.values)
n = len(order)
cut = int(n * 0.8)
train_idx = order[:cut]
test_idx = order[cut:]

raw_train = y_raw_fit[train_idx]
q05, q95 = np.quantile(raw_train, [0.05, 0.95])
span = max(q95 - q05, 1e-6)


def to_lsi_0_100(r: np.ndarray) -> np.ndarray:
    z = (r - q05) / span
    return np.clip(100.0 * z, 0.0, 100.0)


y_all = to_lsi_0_100(y_raw_fit)
y_train = y_all[train_idx]
y_test = y_all[test_idx]
X_train = X_fit.iloc[train_idx]
X_test = X_fit.iloc[test_idx]


def make_pipe(est):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("m", est)])


models = {}

models["hgb_shallow"] = make_pipe(
    HistGradientBoostingRegressor(
        max_depth=5, learning_rate=0.06, max_iter=400, min_samples_leaf=20,
        l2_regularization=0.02, random_state=42,
    )
)
models["hgb_deep"] = make_pipe(
    HistGradientBoostingRegressor(
        max_depth=7, learning_rate=0.04, max_iter=500, min_samples_leaf=12,
        l2_regularization=0.01, random_state=43,
    )
)
models["sklearn_gbr"] = make_pipe(
    GradientBoostingRegressor(
        n_estimators=250, max_depth=4, learning_rate=0.05, min_samples_leaf=15,
        subsample=0.85, random_state=44,
    )
)

try:
    from xgboost import XGBRegressor
    models["xgb"] = make_pipe(
        XGBRegressor(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            random_state=45, n_jobs=-1, verbosity=0,
        )
    )
except Exception as e:
    print("XGBoost пропущен:", e)

try:
    import lightgbm as lgb
    models["lgbm"] = make_pipe(
        lgb.LGBMRegressor(
            n_estimators=400, max_depth=-1, num_leaves=48, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            random_state=46, verbose=-1,
        )
    )
except Exception as e:
    print("LightGBM пропущен:", e)

preds_test = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    p = np.clip(pipe.predict(X_test), 0, 100)
    preds_test[name] = p
    print(f"{name}: test MAE={mean_absolute_error(y_test, p):.3f}, R2={r2_score(y_test, p):.3f}")

P_te = np.column_stack(list(preds_test.values()))
lsi_test = P_te.mean(axis=1)
print("\nАнсамбль (среднее), test: MAE=", round(mean_absolute_error(y_test, lsi_test), 3), " R2=", round(r2_score(y_test, lsi_test), 3))

final_models = {}
for name, pipe in models.items():
    pipe.fit(X_fit, y_all)
    final_models[name] = pipe

stack = [np.clip(pipe.predict(X_fit), 0, 100) for pipe in final_models.values()]
lsi_fit = np.mean(stack, axis=0)

wide_lsi["LSI_teacher"] = np.nan
wide_lsi["LSI_ensemble"] = np.nan
_idx_fit = wide_lsi.loc[present].index
wide_lsi.loc[_idx_fit, "LSI_teacher"] = y_all
wide_lsi.loc[_idx_fit, "LSI_ensemble"] = lsi_fit

wide = wide_lsi.drop(columns=["_raw_stress_teacher"], errors="ignore")
print("\nДобавлены столбцы: LSI_teacher, LSI_ensemble (NaN там, где нет ключевых сигналов).")


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный налог' 'm4_Косвенные налоги' 'm4_НДПИ' 'm4_НДС'
 'm4_НДФЛ' 'm4_НПД' 'm4_Налог на имущество организаций'
 'm4_Налог на имущество физлиц' 'm4_Налог на прибыль' 'm4_Плата за НВОС'
 'm4_Страхование от НС и ПЗ' 'm4_Страховые взносы' 'm4_Торговый сбор'
 'm4_Транспортный налог' 'm4_Туристический налог' 'm4_УСН'
 'm4_Экологический сбор']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный нало

hgb_shallow: test MAE=8.593, R2=0.733


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный налог' 'm4_Косвенные налоги' 'm4_НДПИ' 'm4_НДС'
 'm4_НДФЛ' 'm4_НПД' 'm4_Налог на имущество организаций'
 'm4_Налог на имущество физлиц' 'm4_Налог на прибыль' 'm4_Плата за НВОС'
 'm4_Страхование от НС и ПЗ' 'm4_Страховые взносы' 'm4_Торговый сбор'
 'm4_Транспортный налог' 'm4_Туристический налог' 'm4_УСН'
 'm4_Экологический сбор']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный нало

hgb_deep: test MAE=9.436, R2=0.684


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный налог' 'm4_Косвенные налоги' 'm4_НДПИ' 'm4_НДС'
 'm4_НДФЛ' 'm4_НПД' 'm4_Налог на имущество организаций'
 'm4_Налог на имущество физлиц' 'm4_Налог на прибыль' 'm4_Плата за НВОС'
 'm4_Страхование от НС и ПЗ' 'm4_Страховые взносы' 'm4_Торговый сбор'
 'm4_Транспортный налог' 'm4_Туристический налог' 'm4_УСН'
 'm4_Экологический сбор']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный нало

...

xgb: test MAE=9.720, R2=0.672


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['m4_day' 'm4_m4_day_type_code' 'm4_month' 'm4_n_events_html'
 'm4_n_important' 'm4_year' 'm4_Акцизы' 'm4_Водный налог' 'm4_ЕНП'
 'm4_ЕСХН' 'm4_Земельный налог' 'm4_Косвенные налоги' 'm4_НДПИ' 'm4_НДС'
 'm4_НДФЛ' 'm4_НПД' 'm4_Налог на имущество организаций'
 'm4_Налог на имущество физлиц' 'm4_Налог на прибыль' 'm4_Плата за НВОС'
 'm4_Страхование от НС и ПЗ' 'm4_Страховые взносы' 'm4_Торговый сбор'
 'm4_Транспортный налог' 'm4_Туристический налог' 'm4_УСН'
 'm4_Экологический сбор']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


lgbm: test MAE=9.960, R2=0.660

Ансамбль (среднее), test: MAE= 9.033  R2= 0.712


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Добавлены столбцы: LSI_teacher, LSI_ensemble (NaN там, где нет ключевых сигналов).


In [12]:
# --- Интерпретируемость: усреднённые важности признаков ---
importances = []
for name, pipe in final_models.items():
    m = pipe.named_steps["m"]
    fi = getattr(m, "feature_importances_", None)
    if fi is not None:
        importances.append(np.asarray(fi, dtype=float))

if importances:
    imp_mean = np.mean(importances, axis=0)
    imp_df = pd.DataFrame({"feature": feature_cols, "importance": imp_mean})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("Топ-20 признаков (средняя важность по моделям с feature_importances_):")
    print(imp_df.head(20).to_string(index=False))
else:
    print("Нет feature_importances_ (проверьте версии sklearn/xgboost/lightgbm).")

print("\nПоследние 10 дней с LSI:")
print(wide[["date", "LSI_teacher", "LSI_ensemble"]].dropna().tail(10).to_string(index=False))


Топ-20 признаков (средняя важность по моделям с feature_importances_):
                       feature  importance
                  m1_shift_mad  453.685165
m5_MAD_score_treasury_pressure  376.448780
m5_MAD_score_liquidity_deficit  332.008445
          m5_treasury_pressure  292.380364
                    m1_ruo_mad  281.010347
           Seasonal_Factor_cos  272.000414
            m3_mad_score_cover  262.365008
      m5_treasury_pressure_ma7  249.333461
                        m1_vol  238.000415
                m3_cover_ratio  216.672499
                      m1_shift  214.337150
                        m1_ruo  212.127583
                       m1_fact  197.692500
          m5_liquidity_deficit  189.334369
           Seasonal_Factor_sin  180.666977
  m5_liquidity_deficit_ex_corr  180.333647
              m5_corr_accounts  166.000416
             m5_reserve_buffer  154.666775
             m3_flag_nedospros  136.371271
          m5_required_reserves  129.679254

Последние 10 дней с LSI:


## Прогноз LSI: DLinear ([Time-Series-Library](https://github.com/thuml/Time-Series-Library))

Используется модель **DLinear** (разложение на сезонность и тренд + линейные проекции; см. `models/DLinear.py` в репозитории).

**Данные:** дневной `LSI_ensemble` → **среднемесячный** LSI (`resample('ME')`), чтобы график был **помесячно** за 2020–2025 гг.

**Сплит по времени (без перемешивания):** обучение только на месяцах **строго до 2024-01-01**; прогноз с **walk-forward** на тесте с **2024-01** (история только из прошлого, без утечки).

**Зависимости:** репозиторий `Time-Series-Library` в корне проекта и `torch` (`pip install torch`). При необходимости: `pip install matplotlib`.


In [ ]:
# --- DLinear (TSLib): месячный ряд LSI, обучение до 2024, прогноз с 2024 ---
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from sklearn.preprocessing import StandardScaler
except ImportError:
    raise ImportError("Нужен scikit-learn")

ROOT = Path.cwd().resolve()
TSLIB = ROOT / "Time-Series-Library"
if not TSLIB.is_dir():
    raise FileNotFoundError(
        f"Ожидается клон репозитория: {TSLIB}\n"
        "git clone https://github.com/thuml/Time-Series-Library.git"
    )
if str(TSLIB) not in sys.path:
    sys.path.insert(0, str(TSLIB))

from models.DLinear import Model as DLinearModel

# Дневной ряд → помесячное среднее
if "LSI_ensemble" not in wide.columns:
    raise ValueError("Сначала выполните ячейки с LSI (должен быть столбец LSI_ensemble).")

w = wide.sort_values("date").copy()
w["lsi_d"] = w["LSI_ensemble"].ffill()
daily = w.set_index("date")["lsi_d"].sort_index()
monthly = daily.resample("M").mean()
monthly = monthly.interpolate(method="time").dropna()
monthly = monthly.loc["2010":]  # с начала ряда

SPLIT = pd.Timestamp("2024-01-01")
seq_len = 12
pred_len = 1
if (monthly.index < SPLIT).sum() <= seq_len + 5:
    raise ValueError("Слишком мало месяцев до split для seq_len=12")

train_months = monthly[monthly.index < SPLIT]
test_months = monthly[monthly.index >= SPLIT]

scaler = StandardScaler()
scaler.fit(train_months.values.reshape(-1, 1))
z = pd.Series(
    scaler.transform(monthly.values.reshape(-1, 1)).ravel(),
    index=monthly.index,
    name="lsi_z",
)

# окна обучения: только конец окна < SPLIT
tr = z[z.index < SPLIT]
Xw, yw = [], []
for i in range(seq_len, len(tr)):
    Xw.append(tr.iloc[i - seq_len : i].values)
    yw.append(tr.iloc[i : i + pred_len].values)
X_train = torch.tensor(np.stack(Xw), dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(np.stack(yw), dtype=torch.float32).unsqueeze(-1)

cfg = SimpleNamespace(
    task_name="long_term_forecast",
    seq_len=seq_len,
    pred_len=pred_len,
    enc_in=1,
    dec_in=1,
    c_out=1,
    moving_avg=3,
    num_class=1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ts_model = DLinearModel(cfg).to(device)
opt = torch.optim.Adam(ts_model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

n_ep = 120
bs = 32
N = X_train.shape[0]
ts_model.train()
for ep in range(n_ep):
    perm = torch.randperm(N)
    loss_ep = 0.0
    for s in range(0, N, bs):
        idx = perm[s : s + bs]
        xb = X_train[idx].to(device)
        yb = y_train[idx].to(device)
        opt.zero_grad()
        out = ts_model(xb, None, None, None)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        loss_ep += float(loss.item()) * len(idx)
    loss_ep /= N
    if (ep + 1) % 20 == 0:
        print(f"epoch {ep+1}/{n_ep}  MSE={loss_ep:.6f}")

# Walk-forward прогноз на тесте (месяцы >= SPLIT)
ts_model.eval()
pred_z = []
pred_dates = []
full = z.copy()
for pos in range(len(full)):
    if full.index[pos] < SPLIT:
        continue
    if pos < seq_len:
        continue
    x_in = full.iloc[pos - seq_len : pos].values.astype(np.float32)
    xt = torch.from_numpy(x_in).view(1, seq_len, 1).to(device)
    with torch.no_grad():
        out = ts_model(xt, None, None, None).cpu().numpy().ravel()[:pred_len]
    pred_z.append(out[0])
    pred_dates.append(full.index[pos])

pred_ser = pd.Series(pred_z, index=pd.DatetimeIndex(pred_dates), name="LSI_pred_z")

# Обратное масштабирование → 0–100 по смыслу LSI (как в исходном масштабе)
actual_raw = monthly.copy()
pred_raw = pd.Series(
    scaler.inverse_transform(pred_ser.values.reshape(-1, 1)).ravel(),
    index=pred_ser.index,
    name="LSI_DLinear",
)
pred_raw = pred_raw.clip(lower=0, upper=100)

# Ограничим график 2020–2025
plot_start, plot_end = pd.Timestamp("2020-01-01"), pd.Timestamp("2025-12-31")
mask_plot = (actual_raw.index >= plot_start) & (actual_raw.index <= plot_end)
act_plot = actual_raw.loc[mask_plot].copy()
act_plot = act_plot.clip(lower=0, upper=100)
pred_plot = pred_raw.reindex(act_plot.index)

print("Месяцев на графике (факт):", len(act_plot))
print("Месяцев с прогнозом (>=2024):", pred_plot.notna().sum())


In [ ]:
# --- График: фактический и прогнозный LSI по месяцам (2020–2025) ---
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(act_plot.index, act_plot.values, label="LSI факт (среднее за месяц)", color="steelblue", linewidth=1.8)
ax.plot(pred_plot.index, pred_plot.values, label="LSI DLinear (тест, walk-forward с 2024)", color="darkorange", linewidth=1.5, marker="o", markersize=3)
ax.axvline(SPLIT, color="gray", linestyle="--", linewidth=1, label="Граница train/test (2024-01)")
ax.set_title("Liquidity Stress Index: помесячно, DLinear из Time-Series-Library")
ax.set_ylabel("LSI")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
